# NBA Analyst Agent – Colab runner
Run the cells top to bottom. First add `OPENAI_API_KEY` and `GEMINI_API_KEY` under 🔑 **Secrets** (left sidebar) and turn on notebook access for each.

## 1. Get the code and keys

In [ ]:
!git clone https://github.com/ZachDvorak/basketballm.git
%cd basketballm
!pip install -q -r requirements.txt
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [ ]:
!python src/list_models.py   # note which model names your keys can use

## 2. Convert the Kaggle data
**2a.** Set the folder where your Kaggle CSVs are (check the Colab file browser on the left).

In [ ]:
KAGGLE_DIR = '/content/YOUR-FOLDER-HERE'   # <-- change this
!python src/inspect_csvs.py "$KAGGLE_DIR"

**2b.** Read the output above. Find the CSV with one row per player per game. Then open `data_mapping.example.json` in the file browser (left sidebar), edit it to match your column names, and **save it as `data_mapping.json`** in the same folder. See README section 3 for the rules and the list of required columns.

**2c.** Convert. Read every `NOTE:` line.

In [ ]:
!python src/prepare_data.py --src "$KAGGLE_DIR" --mapping data_mapping.json

## 3. Sanity-check the data and the gold answers

In [ ]:
!python -m pytest tests -q
!python src/evaluate.py --check-gold --limit 10

## 4. Ask the agent something
Change `provider` to `'gemini'` and `model` to a name from the list above.

In [ ]:
import sys; sys.path.insert(0, 'src')
from agent import NBAAgent
agent = NBAAgent(provider='openai', model=None)   # model=None uses the default in src/llm.py
r = agent.ask('Who led the league in points per game in 2024-25 (minimum 50 games)?')
print(r.answer); print(); print(*r.queries, sep='\n---\n')

## 5. Evaluate
Start with `--limit 5` to check cost and plumbing, then run the full set. Compare providers and models with `--provider` / `--model`.

In [ ]:
!python src/evaluate.py --provider openai --limit 5 --judge

In [ ]:
!python src/evaluate.py --provider openai --judge
!python src/evaluate.py --provider gemini --judge

## 6. Chat UI (optional)

In [ ]:
!NBA_PROVIDER=openai python src/app.py